In [1]:
import os
import sys

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset
from PIL import Image, ImageChops
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

In [2]:
# --- Configuration & Setup ---
HF_DATASET = 'TheKernel01/AIGIBench'
IN_COLAB = 'google.colab' in sys.modules
CHECKPOINT_DIR = './checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/HuggingFace/cache'
    os.makedirs(CACHE_DIR, exist_ok=True)
    HF_TOKEN = userdata.get('HF_TOKEN')
else:
    from dotenv import load_dotenv

    load_dotenv()
    CACHE_DIR = None
    HF_TOKEN = os.getenv('HF_TOKEN')

BATCH_SIZE = 32
EPOCHS = 1
IMAGE_SIZE = (224, 224)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
# --- ELA Logic ---
def get_ela_image(image, quality=90):
    """Calculates the Error Level Analysis of a PIL Image."""
    original = image.convert('RGB').resize(IMAGE_SIZE)

    # Create temporary compressed version in memory
    from io import BytesIO

    buffer = BytesIO()
    original.save(buffer, format='JPEG', quality=quality)
    buffer.seek(0)
    compressed = Image.open(buffer)

    # Calculate difference and scale for visibility
    ela_img = ImageChops.difference(original, compressed)
    extrema = ela_img.getextrema()
    max_diff = max([ex[1] for ex in extrema])
    if max_diff == 0:
        max_diff = 1
    scale = 255.0 / max_diff

    # Enhance the difference
    return Image.fromarray(np.uint8(np.array(ela_img) * scale))

In [4]:
class ELADataset(Dataset):
    def __init__(self, hf_dataset, transform=None, use_cache=True):
        self.dataset = hf_dataset
        self.transform = transform
        self.use_cache = use_cache
        self.cache = {}

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        if self.use_cache and idx in self.cache:
            ela_img, label = self.cache[idx]
        else:
            item = self.dataset[idx]
            ela_img = get_ela_image(item['image'])
            label = torch.tensor(item['label'], dtype=torch.long)
            if self.use_cache:
                self.cache[idx] = (ela_img, label)

        if self.transform:
            ela_img = self.transform(ela_img)
        return ela_img, label

In [5]:
# --- Data Loading ---
print(f'Loading dataset: {HF_DATASET}...')
train_data = load_dataset(
    HF_DATASET, token=HF_TOKEN, cache_dir=CACHE_DIR, split='train'
)
val_data = load_dataset(
    HF_DATASET, token=HF_TOKEN, cache_dir=CACHE_DIR, split='validation'
)

transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])]
)

num_workers = min(4, os.cpu_count() or 1)

train_loader = DataLoader(
    ELADataset(train_data, transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
)
val_loader = DataLoader(
    ELADataset(val_data, transform),
    batch_size=BATCH_SIZE,
    num_workers=num_workers,
    pin_memory=True,
)

Loading dataset: TheKernel01/AIGIBench...


Resolving data files:   0%|          | 0/108 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/108 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/104 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/108 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/108 [00:00<?, ?it/s]

In [6]:
# --- Model Definition ---
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2)  # Binary: Real (0) vs Fake (1)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
# --- Training Loop ---
print(f'Starting training on {DEVICE}...')
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 20 == 0:
            print(
                f'Epoch [{epoch + 1}/{EPOCHS}], Step [{i}/{len(train_loader)}], Loss: {loss.item():.4f}'
            )

    # Save Checkpoint
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f'ela_model_epoch_{epoch + 1}.pth')
    torch.save(
        {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': running_loss / len(train_loader),
        },
        checkpoint_path,
    )
    print(f'Checkpoint saved: {checkpoint_path}')


Starting training on cuda...
Epoch [1/1], Step [0/9000], Loss: 0.8265
Epoch [1/1], Step [20/9000], Loss: 0.6164
Epoch [1/1], Step [40/9000], Loss: 0.3060
Epoch [1/1], Step [60/9000], Loss: 0.4402
Epoch [1/1], Step [80/9000], Loss: 0.2519
Epoch [1/1], Step [100/9000], Loss: 0.2519
Epoch [1/1], Step [120/9000], Loss: 0.2302
Epoch [1/1], Step [140/9000], Loss: 0.2455
Epoch [1/1], Step [160/9000], Loss: 0.4735
Epoch [1/1], Step [180/9000], Loss: 0.2012
Epoch [1/1], Step [200/9000], Loss: 0.3354
Epoch [1/1], Step [220/9000], Loss: 0.1536
Epoch [1/1], Step [240/9000], Loss: 0.1612
Epoch [1/1], Step [260/9000], Loss: 0.2064
Epoch [1/1], Step [280/9000], Loss: 0.1381
Epoch [1/1], Step [300/9000], Loss: 0.5791
Epoch [1/1], Step [320/9000], Loss: 0.1780
Epoch [1/1], Step [340/9000], Loss: 0.3642
Epoch [1/1], Step [360/9000], Loss: 0.1229
Epoch [1/1], Step [380/9000], Loss: 0.1488
Epoch [1/1], Step [400/9000], Loss: 0.2754
Epoch [1/1], Step [420/9000], Loss: 0.1587
Epoch [1/1], Step [440/9000], L